In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import re

torch.manual_seed(42)
np.random.seed(42)

# =====================================================================
# 1. ACTUAL TOY TEXT DATASET & TOKENIZER
# =====================================================================

toy_corpus = [
    {
        "context": "Alice is a software engineer living in Berlin and enjoys painting",
        "prompt": "Where does Alice live? Alice lives in",
        "target": "Berlin"
    },
    {
        "context": "Bob is a marine biologist based in Sydney who studies coral reefs",
        "prompt": "What does Bob study? Bob studies",
        "target": "coral"
    },
    {
        "context": "Charlie is an astrophysicist working in Tokyo researching black holes",
        "prompt": "What does Charlie research? Charlie researches",
        "target": "black"
    },
]

# Distractor contexts (unrelated knowledge to overflow STM into LTM)
distractor_contexts = [
    "The Amazon river flows through South America with massive discharge",
    "Mount Everest is the highest mountain peak located in the Himalayas",
    "Quantum computing utilizes quantum superposition and entanglement",
    "Photosynthesis converts carbon dioxide and water into oxygen and glucose"
]

class ToyTextTokenizer:
    """A lightweight word-level tokenizer with special tokens."""
    def __init__(self):
        self.pad_token, self.unk_token = "<pad>", "<unk>"
        self.word2idx = {self.pad_token: 0, self.unk_token: 1}
        self.idx2word = {0: self.pad_token, 1: self.unk_token}

    def fit(self, texts):
        for text in texts:
            for w in re.findall(r"\b\w+\b|[?.,!]", text.lower()):
                if w not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[w] = idx
                    self.idx2word[idx] = w
        return self

    def encode(self, text):
        tokens = re.findall(r"\b\w+\b|[?.,!]", text.lower())
        return [self.word2idx.get(w, self.word2idx[self.unk_token]) for w in tokens]

    def decode(self, token_ids):
        return " ".join([self.idx2word.get(i, self.unk_token) for i in token_ids])

    @property
    def vocab_size(self):
        return len(self.word2idx)

all_texts = [d["context"] for d in toy_corpus] + [d["prompt"] for d in toy_corpus] + [d["target"] for d in toy_corpus] + distractor_contexts
tokenizer = ToyTextTokenizer().fit(all_texts)


# =====================================================================
# 2. M+ ARCHITECTURE (STM + LTM WITH QUERY KEYS & RECALL DECAY)
# =====================================================================

class MPlus(nn.Module):
    def __init__(self, vocab_size, hidden_dim=32, key_dim=16, stm_capacity=8, block_size=4):
        super().__init__()
        self.d = hidden_dim
        self.key_dim = key_dim
        self.stm_capacity = stm_capacity  # Short-Term Memory capacity (tokens)
        self.block_size = block_size      # Memory tokens injected per context

        # Token Embeddings & LayerNorm
        self.token_emb = nn.Embedding(vocab_size, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

        # Memory Injection encoder (distills context into memory tokens)
        self.memory_encoder = nn.Linear(hidden_dim, hidden_dim)

        # Self-Attention projections
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

        # Key selector for LTM associative retrieval
        self.key_selector = nn.Linear(hidden_dim, key_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

        # 1. Short-Term Memory (STM)
        self.register_buffer("stm", torch.zeros(stm_capacity, hidden_dim))
        self.stm_ages = np.zeros(stm_capacity, dtype=int)

        # 2. Long-Term Memory (LTM)
        self.reset_ltm()

    def reset_ltm(self):
        self.ltm_vectors = []        # Vector representations: [d]
        self.ltm_keys = []           # Retrieval keys: [key_dim]
        self.ltm_rf = []             # Recall frequencies
        self.ltm_ages = []           # Memory token ages

    def compute_keys(self, hiddens):
        return self.key_selector(hiddens)

    def extract_delta_memory(self, context_ids):
        """Passes context through encoder to produce delta memory."""
        x = self.norm(self.token_emb(context_ids))
        h = torch.relu(self.memory_encoder(x))
        delta = h[0, :self.block_size]
        if delta.size(0) < self.block_size:
            pad = torch.zeros(self.block_size - delta.size(0), self.d, device=context_ids.device)
            delta = torch.cat([delta, pad], dim=0)
        return delta

    def inject_memory(self, context_ids, update_memory=True):
        """
        Injects context into STM and migrates overflow into LTM.
        Zero gradient updates on weights during injection!
        """
        delta = self.extract_delta_memory(context_ids)
        num_delta = delta.size(0)

        if update_memory:
            # Oldest STM tokens are evicted
            dropped_memories = self.stm[:num_delta].clone().detach()
            dropped_keys = self.compute_keys(dropped_memories).detach()
            dropped_ages = self.stm_ages[:num_delta].copy()

            # Append new delta into STM
            self.stm = torch.cat([self.stm[num_delta:], delta.detach()], dim=0)
            self.stm_ages = np.concatenate([self.stm_ages[num_delta:] + num_delta, np.zeros(num_delta, dtype=int)])

            # Migrate evicted memories into LTM
            self.update_ltm(dropped_memories, dropped_keys, dropped_ages)

        return delta

    def update_ltm(self, memories, keys, ages, initial_rf=1.0, decay_rate=0.2):
        """Stores evicted STM items into LTM and applies decay & pruning."""
        for i in range(memories.size(0)):
            self.ltm_vectors.append(memories[i].cpu())
            self.ltm_keys.append(keys[i].cpu())
            self.ltm_rf.append(initial_rf)
            self.ltm_ages.append(int(ages[i]))

        # Apply decay to all LTM entries
        for i in range(len(self.ltm_rf)):
            self.ltm_rf[i] -= decay_rate
            self.ltm_ages[i] += 1

        # Prune stale memories where RF <= 0.01
        kept_v, kept_k, kept_rf, kept_ages = [], [], [], []
        for v, k, rf, age in zip(self.ltm_vectors, self.ltm_keys, self.ltm_rf, self.ltm_ages):
            if rf > 0.01:
                kept_v.append(v)
                kept_k.append(k)
                kept_rf.append(rf)
                kept_ages.append(age)
        self.ltm_vectors, self.ltm_keys, self.ltm_rf, self.ltm_ages = kept_v, kept_k, kept_rf, kept_ages

    def get_ltm(self, query_hiddens, top_k=2):
        """Retrieves top_k relevant LTM vectors via query-key dot products."""
        if len(self.ltm_vectors) == 0:
            return torch.empty(0, self.d, device=query_hiddens.device)

        k = min(top_k, len(self.ltm_vectors))
        ltm_keys_t = torch.stack(self.ltm_keys).to(query_hiddens.device)
        queries = self.compute_keys(query_hiddens[0])

        sim_scores = torch.sigmoid(queries @ ltm_keys_t.T).mean(dim=0)
        top_indices = torch.topk(sim_scores, k=k).indices.cpu().numpy()

        for idx in top_indices:
            self.ltm_rf[idx] += 1.0  # Reinforce retrieved memory

        retrieved = torch.stack([self.ltm_vectors[idx] for idx in top_indices]).to(query_hiddens.device)
        return retrieved

    def forward(self, prompt_ids, top_k_ltm=2, explicit_memory=None):
        x = self.norm(self.token_emb(prompt_ids))

        if explicit_memory is not None:
            # During training: cross-attend directly over training memory tokens
            full_context = torch.cat([explicit_memory.unsqueeze(0), x], dim=1)
        else:
            # During inference: retrieve from LTM, concatenate [LTM, STM, Prompt]
            retrieved_ltm = self.get_ltm(x, top_k=top_k_ltm)
            if retrieved_ltm.size(0) > 0:
                full_context = torch.cat([retrieved_ltm.unsqueeze(0), self.stm.unsqueeze(0), x], dim=1)
            else:
                full_context = torch.cat([self.stm.unsqueeze(0), x], dim=1)

        # Multi-Head Attention
        Q = self.q_proj(full_context)
        K = self.k_proj(full_context)
        V = self.v_proj(full_context)
        attn = torch.softmax((Q @ K.transpose(-2, -1)) / (self.d ** 0.5), dim=-1)
        out = attn @ V

        # Predict next token from the final prompt position
        return self.lm_head(out[:, -1, :])


# =====================================================================
# 3. PHASE 1: TRAINING ON TOY TEXT DATASET
# =====================================================================

model = MPlus(vocab_size=tokenizer.vocab_size, hidden_dim=32, key_dim=16, stm_capacity=8, block_size=4)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.01)

print("=" * 65)
print("PHASE 1: TRAINING BASE MODEL ON TOY TEXT DATASET")
print("Teaches model how to write context into memory & read from memory")
print("=" * 65)

model.train()
for epoch in range(1, 36):
    total_loss = 0.0
    for sample in toy_corpus:
        optimizer.zero_grad()
        ctx_ids = torch.tensor([tokenizer.encode(sample["context"])])
        prompt_ids = torch.tensor([tokenizer.encode(sample["prompt"])])
        target_id = torch.tensor([tokenizer.encode(sample["target"])[0]])

        delta_mem = model.extract_delta_memory(ctx_ids)
        logits = model(prompt_ids, explicit_memory=delta_mem)

        loss = criterion(logits, target_id)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 7 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/35 | Training Loss: {total_loss / len(toy_corpus):.4f}")


# =====================================================================
# 4. PHASE 2: LIFELONG INFERENCE (ZERO GRADIENTS / WEIGHTS FROZEN)
# =====================================================================

print("\n" + "=" * 65)
print("PHASE 2: LIFELONG INFERENCE WITH M+ MEMORY HIERARCHY")
print("All model weights are now STRICTLY FROZEN.")
print("=" * 65)

model.eval()
for param in model.parameters():
    param.requires_grad = False

model.stm.zero_()
model.reset_ltm()

# Inject Fact A: "Alice lives in Berlin"
fact_A = toy_corpus[0]
print(f"\n[1] Injecting Fact A: \"{fact_A['context']}\"")
model.inject_memory(torch.tensor([tokenizer.encode(fact_A["context"])]), update_memory=True)
print(f"    STM count: {model.stm.shape[0]} | LTM count: {len(model.ltm_vectors)}")

# Inject distractors to overflow STM and migrate Fact A into LTM
print("\n[2] Injecting distractor texts to push Fact A into LTM...")
for d_text in distractor_contexts[:2]:
    print(f"    Injecting distractor: \"{d_text[:40]}...\"")
    model.inject_memory(torch.tensor([tokenizer.encode(d_text)]), update_memory=True)

print(f"    -> Fact A has been migrated from STM into Long-Term Memory (LTM).")
print(f"    -> LTM count: {len(model.ltm_vectors)} tokens.")

# Query the model on Fact A (retrieved entirely from LTM)
print("\n" + "-" * 65)
print("TEST: Querying Fact A (Retrieved from Long-Term Memory)")
prompt_A = fact_A["prompt"]
p_ids = torch.tensor([tokenizer.encode(prompt_A)])
logits = model(p_ids, top_k_ltm=2)

pred_id = torch.argmax(logits, dim=-1).item()
pred_word = tokenizer.decode([pred_id])

print(f"Input Prompt: \"{prompt_A}\"")
print(f"Expected Target: \"{fact_A['target']}\"")
print(f"Model Prediction: \"{pred_word}\" (Match: {pred_word.lower() == fact_A['target'].lower()})")
print("=" * 65)


PHASE 1: TRAINING BASE MODEL ON TOY TEXT DATASET
Teaches model how to write context into memory & read from memory
Epoch  1/35 | Training Loss: 4.3094
Epoch  7/35 | Training Loss: 0.0020
Epoch 14/35 | Training Loss: 0.0001
Epoch 21/35 | Training Loss: 0.0001
Epoch 28/35 | Training Loss: 0.0001
Epoch 35/35 | Training Loss: 0.0001

PHASE 2: LIFELONG INFERENCE WITH M+ MEMORY HIERARCHY
All model weights are now STRICTLY FROZEN.

[1] Injecting Fact A: "Alice is a software engineer living in Berlin and enjoys painting"
    STM count: 8 | LTM count: 4

[2] Injecting distractor texts to push Fact A into LTM...
    Injecting distractor: "The Amazon river flows through South Ame..."
    Injecting distractor: "Mount Everest is the highest mountain pe..."
    -> Fact A has been migrated from STM into Long-Term Memory (LTM).
    -> LTM count: 12 tokens.

-----------------------------------------------------------------
TEST: Querying Fact A (Retrieved from Long-Term Memory)
Input Prompt: "Where doe